# 원하는 포즈로 이미지 만드는 도구

참조 사진에서 사람의 관절(포즈)을 뽑아, **그 자세 그대로** 다른 인물·장면을 만든다.

수업에서 쓴 **FLUX.2-klein 4B (GGUF Q4) + ComfyUI** 노트를 출발점으로 삼아, 거기에 포즈 조건을 얹었다.

---

## 어떻게 돌아가나

```
참조 사진  →  OpenPose 관절 추출  →  뼈대 그림
                                        ↓
                                   VAE 인코딩
                                        ↓
프롬프트 → 텍스트 인코더 →  ReferenceLatent  →  klein 4B  →  결과 이미지
```

뼈대 그림을 **참조 이미지**로 넣으면, 모델이 그 자세를 따라 그린다.

## 왜 ControlNet 을 안 쓰나

2026년 9월 기준 FLUX.2-klein **4B 전용** ControlNet 이 없다. 나와 있는 포즈용 파일은 전부 9B 용이거나 다른 모델용이다.

| 파일 | 대상 모델 |
|---|---|
| `Flux.2-Klein-9B-MatchingPose` | klein 9B |
| `refcontrol-FLUX.2-klein-9B-reference-pose-lora` | klein 9B |
| `alibaba-pai/FLUX.2-dev-Fun-Controlnet-Union` | FLUX.2-dev |

대신 klein 은 **텍스트→이미지와 이미지→이미지를 한 모델에 합쳐 놓은** 구조라 참조 이미지를 직접 받을 수 있다. 그래서 별도 ControlNet 없이 ComfyUI 기본 노드 `ReferenceLatent` 로 뼈대 그림을 꽂아 넣는다.

## 수업 노트에서 바뀐 곳

| | 수업 노트 | 이 노트 |
|---|---|---|
| 설치 | ComfyUI + GGUF | **+ `comfyui_controlnet_aux`** (포즈 추출기) |
| 입력 | 프롬프트만 | **프롬프트 + 참조 사진** |
| 워크플로 | 빈 캔버스에서 생성 | **뼈대 그림을 참조로 투입** |
| 시드 | 매번 임의 | **고정 + 설정 기록** (재현용) |

---

## 실행 전 준비

1. 참조로 쓸 사람 사진을 저장소의 `samples/pose_01.png`, `samples/pose_02.png` 로 올려 둔다 (셀 6 이 이 주소에서 내려받는다)
2. GPU 런타임에 연결한다
   - **VS Code**: 오른쪽 위 `커널 선택` → `다른 커널 선택` → `Colab` → 구글 로그인 → **T4 GPU** 로 새 서버 만들기
   - **브라우저 Colab**: 상단 `런타임` → `런타임 유형 변경` → **T4 GPU**
3. 셀을 **1번부터 차례대로** 실행 (모델 약 11GB 내려받느라 처음엔 시간이 걸린다)
4. 실험할 때 고치는 곳은 **[셀 5] 설정** 한 군데뿐이다

| 셀 | 하는 일 |
|---|---|
| 1 | GPU 확인 |
| 2 | ComfyUI + 포즈 추출기 설치 |
| 3 | 모델 3종 내려받기 |
| 4 | **모델 로드** (ComfyUI 서버 기동) |
| 5 | 설정 — 여기만 고친다 |
| 6 | 참조 사진 가져오기 |
| 7 | **포즈 추출** |
| 8 | **이미지 생성** |
| 9 | **결과 저장** |
| 10 | 결과 내 PC 로 가져오기 |
| 11 | (선택) 버튼 달린 UI |

## [셀 1] GPU 확인

이 도구는 GPU 없이는 돌아가지 않는다. 런타임이 GPU 를 잡고 있는지, 어떤 종류이고 메모리는 얼마인지 확인한다.

In [ ]:
# [셀 1] GPU 가 잡혔는지 확인한다. 안 잡히면 [런타임 → 런타임 유형 변경 → T4 GPU] 후 다시 실행한다.
import torch

print("PyTorch:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM(GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    raise RuntimeError("GPU가 없습니다. [런타임 → 런타임 유형 변경 → T4 GPU] 후 다시 실행하세요.")

## [셀 2] ComfyUI 와 플러그인 2개 설치

**ComfyUI** 는 기능 블록(노드)을 연결해 이미지 파이프라인을 만드는 도구다. 여기서는 웹 화면 없이 **API 서버로만** 쓴다.

여기에 플러그인 두 개를 더한다.

| 플러그인 | 왜 필요한가 |
|---|---|
| `ComfyUI-GGUF` | 용량을 줄인 GGUF 모델을 읽기 위해 (기본 기능으로는 못 읽는다) |
| `comfyui_controlnet_aux` | **사진에서 사람 관절을 뽑아내기 위해** (OpenPose) |

아래쪽이 수업 노트에 없던 부분이다.

In [ ]:
# [셀 2] ComfyUI 본체 + 플러그인 2개(GGUF 로더, 포즈 추출기)를 설치한다. 런타임당 한 번만 하면 된다.
import os

%cd /content
if not os.path.isdir("/content/ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git

%cd /content/ComfyUI
!pip install -q -r requirements.txt

%cd /content/ComfyUI/custom_nodes

# (1) GGUF 포맷을 읽는 로더
if not os.path.isdir("ComfyUI-GGUF"):
    !git clone https://github.com/city96/ComfyUI-GGUF.git

# (2) 사진에서 관절을 뽑는 전처리기 — 이 노트에서 새로 추가한 부분
if not os.path.isdir("comfyui_controlnet_aux"):
    !git clone https://github.com/Fannovel16/comfyui_controlnet_aux.git
!pip install -q -r comfyui_controlnet_aux/requirements.txt

%cd /content/ComfyUI
!pip install -q gguf

print("\n완료: ComfyUI + GGUF 로더 + 포즈 추출기")

## [셀 3] 모델 3종 내려받기 (약 11GB)

이 파이프라인은 파일 세 개가 모여야 돌아간다.

| 역할 | 파일 | 크기 |
|---|---|---|
| **그림 그리는 본체** | `flux-2-klein-4b-Q4_K_M.gguf` | 2.6GB |
| **프롬프트 이해** | `qwen_3_4b.safetensors` | 8GB |
| **latent → 픽셀 변환** | `flux2-vae.safetensors` | 330MB |

본체가 2.6GB 밖에 안 되는 건 **양자화** 덕분이다. 원래 16GB 인 모델의 숫자 정밀도를 낮춰 크기를 줄인 것으로, 무료 Colab 메모리에 들어가게 하려는 조치다.

이미 받은 파일은 건너뛰므로, 런타임이 끊겨 다시 실행해도 처음부터 받지 않는다.

In [ ]:
# [셀 3] 모델 파일 3종을 ComfyUI 가 찾는 표준 폴더에 내려받는다. 이미 있으면 건너뛴다.
import os
%cd /content/ComfyUI

os.makedirs("models/diffusion_models", exist_ok=True)
os.makedirs("models/text_encoders", exist_ok=True)
os.makedirs("models/vae", exist_ok=True)

def 다운로드(url, 저장경로):
    """이미 받았으면 건너뛰고, 아니면 받는다."""
    if os.path.exists(저장경로) and os.path.getsize(저장경로) > 1_000_000:
        print(f"  ↳ 이미 있음: {저장경로} ({os.path.getsize(저장경로)/1e9:.2f} GB)")
        return
    print("  ↳ 내려받는 중...")
    !wget -q --show-progress -O "{저장경로}" "{url}"

print("[1/3] 그림 그리는 본체 (GGUF Q4, 약 2.6GB)")
다운로드("https://huggingface.co/unsloth/FLUX.2-klein-4B-GGUF/resolve/main/flux-2-klein-4b-Q4_K_M.gguf",
        "models/diffusion_models/flux-2-klein-4b-Q4_K_M.gguf")

print("[2/3] 텍스트 인코더 (Qwen3-4B, 약 8GB)")
다운로드("https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-4b/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors",
        "models/text_encoders/qwen_3_4b.safetensors")

print("[3/3] VAE (약 330MB)")
다운로드("https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-4b/resolve/main/split_files/vae/flux2-vae.safetensors",
        "models/vae/flux2-vae.safetensors")

print("\n배치된 파일:")
!ls -lh models/diffusion_models/flux-2-klein-4b-Q4_K_M.gguf \
      models/text_encoders/qwen_3_4b.safetensors \
      models/vae/flux2-vae.safetensors

## [셀 4] 모델 로드 — ComfyUI 서버 기동

ComfyUI 를 백그라운드 서버로 띄운다. 이 서버가 모델을 메모리에 올리고, 앞으로 모든 작업 요청을 받는다. 주소는 Colab 컨테이너 안의 `http://127.0.0.1:8188` 이다.

서버에 작업을 보내는 **헬퍼 함수 `comfy_run()`** 도 여기서 만들어 둔다. 이 함수 하나를 셀 7(포즈 추출)과 셀 8(이미지 생성)이 함께 쓴다.

`http 200` 이 확인되면 준비 완료다.

In [ ]:
# [셀 4] ComfyUI 서버를 백그라운드로 띄우고, 작업을 보내는 헬퍼 함수를 준비한다.
import subprocess, os, time, json, io, urllib.parse, shutil
import requests
from PIL import Image

%cd /content/ComfyUI

SERVER = "http://127.0.0.1:8188"
INPUT_DIR = "/content/ComfyUI/input"   # ComfyUI 는 여기 있는 파일만 불러올 수 있다
os.makedirs(INPUT_DIR, exist_ok=True)

# 예전에 뜬 서버가 있으면 정리
!pkill -f "main.py" 2>/dev/null
time.sleep(2)

os.system("nohup python main.py --listen 0.0.0.0 --port 8188 > /content/comfyui.log 2>&1 &")

print("ComfyUI 기동 대기 중...")
for _ in range(60):
    time.sleep(1)
    code = subprocess.run("curl -s -o /dev/null -w '%{http_code}' http://127.0.0.1:8188/",
                          shell=True, capture_output=True, text=True).stdout
    if code == "200":
        print("완료: ComfyUI 서버 준비")
        break
else:
    print("서버가 안 뜹니다. 로그 끝부분:")
    print(open("/content/comfyui.log").read()[-1500:])


def comfy_run(workflow, 결과노드, 제한초=600):
    """워크플로(노드 그래프)를 ComfyUI 에 보내고, 끝나면 결과 이미지를 PIL 로 돌려준다.

    workflow : {노드번호: {class_type, inputs}} 형태의 딕셔너리
    결과노드 : 이미지를 꺼내 올 SaveImage 노드의 번호(문자열)
    """
    r = requests.post(f"{SERVER}/prompt", json={"prompt": workflow, "client_id": "colab"}, timeout=60)
    if r.status_code != 200:
        raise RuntimeError(f"요청 거부 {r.status_code}: {r.text[:800]}")
    pid = r.json()["prompt_id"]

    t0 = time.time()
    while time.time() - t0 < 제한초:
        time.sleep(2)
        try:
            h = requests.get(f"{SERVER}/history/{pid}", timeout=15).json()
        except Exception:
            continue
        if pid not in h:
            continue
        entry = h[pid]
        if entry.get("status", {}).get("status_str") == "error":
            raise RuntimeError("ComfyUI 에러: " +
                               json.dumps(entry["status"].get("messages", []), ensure_ascii=False)[:1500])
        outs = entry.get("outputs", {})
        if 결과노드 not in outs:
            continue
        이미지들 = []
        for img in outs[결과노드].get("images", []):
            q = urllib.parse.urlencode({"filename": img["filename"],
                                        "subfolder": img.get("subfolder", ""),
                                        "type": img.get("type", "output")})
            rr = requests.get(f"{SERVER}/view?{q}", timeout=60)
            이미지들.append(Image.open(io.BytesIO(rr.content)).convert("RGB"))
        if 이미지들:
            print(f"  ({round(time.time()-t0,1)}초)")
            return 이미지들
    raise TimeoutError("시간 초과")

print("헬퍼 준비 완료: comfy_run()")

## [셀 5] 설정 — 실험할 때 고치는 곳은 여기뿐

아래 값만 바꿔 가며 실험한다. 다른 셀은 건드리지 않는다.

**재현성(루브릭 2번)의 핵심은 `SEED` 다.** 같은 시드 + 같은 설정 = 같은 결과. 시드를 고정하지 않으면 매번 다른 그림이 나와서 "똑같이 다시 만들어 보라"가 불가능해진다.

| 값 | 뜻 | 바꿔 볼 만한 범위 |
|---|---|---|
| `PROMPT` | 어떤 장면을 만들지 | 자유 (영어가 잘 먹는다) |
| `SEED` | 난수 고정값 | 아무 정수. **고정해 둘 것** |
| `STEPS` | 그리는 횟수 | 4 (이 모델은 4회용으로 압축돼 있다) |
| `CFG` | 프롬프트를 얼마나 강하게 따를지 | 1.5 ~ 4.0 |
| `참조모드` | 무엇을 참조로 넣을지 | `"뼈대"` 또는 `"원본"` |
| `참조사진_URL` | 자세를 가져올 사진 주소 | 저장소의 `samples/pose_XX.png` |
| `RUN_ID` | 결과 번호 | `"01"`, `"02"`, ... |

`참조모드` 는 이 노트의 관찰 포인트다. `"뼈대"` 는 관절만 넘겨서 인물·배경을 자유롭게 바꾸고, `"원본"` 은 사진 자체를 넘겨서 자세는 잘 따라오지만 원본 인물의 생김새까지 묻어 나온다.

In [ ]:
# [셀 5] 설정. 실험할 때 고치는 곳은 여기뿐이다.
from pathlib import Path

# --- 만들고 싶은 장면 ---
PROMPT = "an astronaut in a white spacesuit, jumping with arms and legs spread wide, photorealistic, studio lighting, plain background"
NEGATIVE = "lowres, bad anatomy, extra limbs, deformed hands, worst quality"

# --- 재현을 위한 고정값 (루브릭 2번) ---
SEED = 20260920      # 같은 시드 + 같은 설정 = 같은 결과
STEPS = 4            # 이 모델은 4스텝용으로 압축돼 있다
CFG = 2.5            # 프롬프트를 얼마나 강하게 따를지
WIDTH, HEIGHT = 1024, 1536

# --- 포즈를 어떻게 넘길지 ---
참조모드 = "뼈대"     # "뼈대" = OpenPose 관절만 / "원본" = 사진 그대로

# --- 결과 번호. 실험할 때마다 올린다 (output_01, output_02 ...) ---
RUN_ID = "01"

# --- 자세를 가져올 참조 사진 (저장소에 올려 둔 것을 주소로 불러온다) ---
저장소 = "https://raw.githubusercontent.com/ctaleez51-art/pose-image-tool/main/samples"
참조사진_URL = f"{저장소}/pose_{RUN_ID}.png"

OUT_DIR = Path("/content/samples")
OUT_DIR.mkdir(exist_ok=True, parents=True)

print(f"RUN_ID={RUN_ID} / SEED={SEED} / STEPS={STEPS} / CFG={CFG} / 참조모드={참조모드}")
print(f"참조 사진: {참조사진_URL}")

## [셀 6] 참조 사진 가져오기

자세를 가져올 사람 사진을 불러온다. **GitHub 저장소에 올려 둔 사진을 주소로 내려받는** 방식이다.

파일 선택 창을 띄우지 않고 주소로 받는 이유는 두 가지다.

1. VS Code 에서 Colab 런타임에 붙여 쓰면 구글 쪽 컴퓨터가 내 PC 의 파일을 보지 못한다. `files.upload()` 가 동작하지 않는다.
2. 주소로 받으면 **누가 언제 실행해도 똑같은 사진**이 들어간다. 재현성(루브릭 2번)에 유리하다.

그래서 참조 사진은 미리 저장소의 `samples/` 에 올려 둔다. 저장소가 **Public** 이어야 한다.

**좋은 참조 사진의 조건**

- 전신이 나오고 손발이 잘리지 않았을 것
- 배경이 단순할 것 (관절 인식이 훨씬 잘 된다)
- 사람이 한 명일 것
- 자세가 뚜렷할 것 (애매하면 포즈가 맞았는지 판정하기 어렵다)

In [ ]:
# [셀 6] 자세를 가져올 참조 사진을 주소에서 내려받는다.
from IPython.display import display

r = requests.get(참조사진_URL, timeout=60)
if r.status_code != 200:
    raise RuntimeError(
        f"사진을 못 받았습니다 (HTTP {r.status_code}).\n"
        f"  주소: {참조사진_URL}\n"
        f"  확인할 것: 저장소에 samples/pose_{RUN_ID}.png 가 올라가 있는가 / 저장소가 Public 인가"
    )

원본 = Image.open(io.BytesIO(r.content)).convert("RGB")

# 제출 규격에 맞춰 samples/pose_XX.png 로 저장
원본경로 = OUT_DIR / f"pose_{RUN_ID}.png"
원본.save(원본경로)

# ComfyUI 는 자기 input 폴더에 있는 파일만 읽을 수 있으므로 거기에도 복사
COMFY_원본 = f"ref_{RUN_ID}.png"
원본.save(os.path.join(INPUT_DIR, COMFY_원본))

print(f"불러옴: {참조사진_URL}")
print(f"크기: {원본.width} x {원본.height}")
display(원본.resize((원본.width // 3, 원본.height // 3)))

## [셀 7] 포즈 추출 — OpenPose 로 관절 뽑기

사진에서 사람의 관절 위치를 찾아 **뼈대 그림**(막대 인간)으로 그려낸다. 이것이 다음 셀에서 모델에게 넘어갈 "이 자세로 그려라"라는 조건이다.

검은 배경에 색색의 막대와 점이 나오면 성공이다. 관절이 어긋나거나 팔다리가 빠졌다면 참조 사진을 바꾸는 게 빠르다.

처음 실행할 때는 관절 인식 모델을 내려받느라 1~2분 걸린다.

In [ ]:
# [셀 7] 사진에서 OpenPose 로 관절을 뽑아 '뼈대 그림'을 만든다. 이것이 모델에 넘어갈 포즈 조건이다.

포즈_워크플로 = {
    # 참조 사진 불러오기
    "1": {"class_type": "LoadImage",
          "inputs": {"image": COMFY_원본}},
    # 관절 추출 (OpenPose)
    "2": {"class_type": "OpenposePreprocessor",
          "inputs": {"image": ["1", 0],
                     "detect_body": "enable",
                     "detect_hand": "enable",
                     "detect_face": "disable",
                     "resolution": 1024}},
    # 뼈대 그림 저장
    "3": {"class_type": "SaveImage",
          "inputs": {"images": ["2", 0], "filename_prefix": "skeleton"}},
}

print("관절 추출 중...")
뼈대 = comfy_run(포즈_워크플로, 결과노드="3")[0]

# 뼈대 그림을 저장하고, ComfyUI 가 다시 읽을 수 있게 input 폴더에도 둔다
뼈대경로 = OUT_DIR / f"pose_{RUN_ID}_skeleton.png"
뼈대.save(뼈대경로)

COMFY_뼈대 = f"skeleton_{RUN_ID}.png"
뼈대.save(os.path.join(INPUT_DIR, COMFY_뼈대))

print(f"뼈대 크기: {뼈대.width} x {뼈대.height}")
print(f"저장: {뼈대경로}")
display(뼈대.resize((뼈대.width // 3, 뼈대.height // 3)))

## [셀 8] 이미지 생성

뼈대 그림과 프롬프트를 함께 넣어 그림을 만든다. 노드 연결은 이렇다.

```
뼈대 그림 ──→ VAEEncode ──┐
                          ├──→ ReferenceLatent ──→ KSampler ──→ VAEDecode ──→ 결과
프롬프트 ──→ CLIPTextEncode ┘                          ↑
                                        klein 4B (UnetLoaderGGUF)
```

핵심은 **`ReferenceLatent`** 다. 프롬프트로 만든 조건에 "이 그림도 참고해라"를 덧붙이는 노드로, ComfyUI 기본 내장이다. klein 이 이미지 입력을 원래 받을 수 있기 때문에 ControlNet 없이 이걸로 된다.

첫 실행은 모델을 메모리에 올리느라 오래 걸린다(2~4분). 두 번째부터는 빨라진다.

In [ ]:
# [셀 8] 포즈 조건 + 프롬프트로 이미지를 만든다. 시드가 고정돼 있어 같은 설정이면 같은 결과가 나온다.

참조파일 = COMFY_뼈대 if 참조모드 == "뼈대" else COMFY_원본

생성_워크플로 = {
    # ---- 모델 3종 로드 ----
    "10": {"class_type": "UnetLoaderGGUF",
           "inputs": {"unet_name": "flux-2-klein-4b-Q4_K_M.gguf"}},
    "8":  {"class_type": "CLIPLoader",
           "inputs": {"clip_name": "qwen_3_4b.safetensors", "type": "flux2"}},
    "4":  {"class_type": "VAELoader",
           "inputs": {"vae_name": "flux2-vae.safetensors"}},

    # ---- 포즈 조건: 참조 그림을 latent 로 바꿔 프롬프트에 덧붙인다 ----
    "12": {"class_type": "LoadImage",
           "inputs": {"image": 참조파일}},
    "14": {"class_type": "VAEEncode",
           "inputs": {"pixels": ["12", 0], "vae": ["4", 0]}},

    # ---- 프롬프트 인코딩 ----
    "6":  {"class_type": "CLIPTextEncode",
           "inputs": {"text": PROMPT, "clip": ["8", 0]}},
    "15": {"class_type": "ReferenceLatent",
           "inputs": {"conditioning": ["6", 0], "latent": ["14", 0]}},
    "7":  {"class_type": "CLIPTextEncode",
           "inputs": {"text": NEGATIVE, "clip": ["8", 0]}},

    # ---- 생성 ----
    "5":  {"class_type": "EmptyLatentImage",
           "inputs": {"width": WIDTH, "height": HEIGHT, "batch_size": 1}},
    "3":  {"class_type": "KSampler",
           "inputs": {"seed": SEED, "steps": STEPS, "cfg": CFG,
                      "sampler_name": "euler", "scheduler": "simple", "denoise": 1.0,
                      "model": ["10", 0], "positive": ["15", 0],
                      "negative": ["7", 0], "latent_image": ["5", 0]}},
    "9":  {"class_type": "VAEDecode",
           "inputs": {"samples": ["3", 0], "vae": ["4", 0]}},
    "11": {"class_type": "SaveImage",
           "inputs": {"images": ["9", 0], "filename_prefix": "pose_out"}},
}

print(f"생성 중... (참조: {참조모드})")
결과 = comfy_run(생성_워크플로, 결과노드="11")[0]
display(결과.resize((결과.width // 2, 결과.height // 2)))

## [셀 9] 결과 저장 + 설정 기록

결과를 제출 규격(`output_XX.png`)으로 저장하고, **무슨 설정으로 만든 것인지 같이 적어 둔다.**

이 기록이 루브릭 2번(재현성)의 증거다. 나중에 같은 그림을 다시 만들려면 이 파일을 보고 셀 5의 값을 그대로 넣으면 된다.

옆에 참조 사진·뼈대·결과를 나란히 붙인 비교 그림도 만든다. 포즈가 맞았는지 한눈에 보라고 만드는 것이고, 채점자도 이걸 본다.

In [ ]:
# [셀 9] 결과를 제출 규격으로 저장하고, 재현에 필요한 설정을 함께 기록한다.

결과경로 = OUT_DIR / f"output_{RUN_ID}.png"
결과.save(결과경로)

기록 = {
    "run_id": RUN_ID,
    "프롬프트": PROMPT,
    "네거티브": NEGATIVE,
    "시드": SEED,
    "스텝": STEPS,
    "cfg": CFG,
    "해상도": f"{WIDTH}x{HEIGHT}",
    "참조모드": 참조모드,
    "샘플러": "euler / simple",
    "모델": "flux-2-klein-4b-Q4_K_M.gguf",
    "텍스트인코더": "qwen_3_4b.safetensors",
    "vae": "flux2-vae.safetensors",
    "포즈추출": "OpenposePreprocessor (comfyui_controlnet_aux)",
    "포즈투입": "ReferenceLatent (ControlNet 미사용)",
}
(OUT_DIR / f"output_{RUN_ID}.json").write_text(
    json.dumps(기록, ensure_ascii=False, indent=2), encoding="utf-8")

# 참조 / 뼈대 / 결과를 나란히 붙인 비교 그림
높이 = 640
def 맞추기(im):
    비율 = 높이 / im.height
    return im.resize((int(im.width * 비율), 높이))

조각 = [맞추기(원본), 맞추기(뼈대), 맞추기(결과)]
비교 = Image.new("RGB", (sum(i.width for i in 조각), 높이), "white")
x = 0
for i in 조각:
    비교.paste(i, (x, 0))
    x += i.width
비교.save(OUT_DIR / f"compare_{RUN_ID}.png")

print(f"저장 완료: {결과경로}")
print(json.dumps(기록, ensure_ascii=False, indent=2))
print("\n왼쪽부터: 참조 사진 / 뼈대 / 결과")
display(비교.resize((비교.width // 2, 높이 // 2)))

## [셀 10] 결과 내 PC 로 가져오기

Colab 은 남의 컴퓨터라, 여기서 만든 파일은 **내 PC 로 가져와야** 저장소에 올릴 수 있다. 이 셀을 건너뛰면 `samples/` 가 빈 채로 제출된다.

결과를 `samples.zip` 하나로 묶은 다음, 아래 세 가지 중 되는 방법으로 가져온다.

| 방법 | 쓰는 곳 | 어떻게 |
|---|---|---|
| **A. 파일 탐색기** | VS Code | 왼쪽 Colab 파일 목록에서 `samples.zip` 우클릭 → 다운로드 |
| **B. 구글 드라이브** | 어디서나 (가장 확실) | 아래 드라이브 칸의 `#` 를 지우고 실행 → 드라이브에서 받기 |
| **C. 자동 다운로드** | 브라우저 Colab 전용 | 자동 시도. VS Code 에서는 실패한다 |

VS Code 에서는 A 가 안 될 때가 있다(구글이 알려진 문제로 등록해 둔 사안). 그럴 땐 B 로 간다.

**실험을 다 마친 뒤 마지막에 한 번만** 실행하는 게 편하다.

In [ ]:
# [셀 10] 지금까지 만든 결과를 zip 으로 묶어 내 PC 로 가져온다.

print("담긴 파일:")
for p in sorted(OUT_DIR.iterdir()):
    print(f"  {p.name}  ({p.stat().st_size/1024:.0f} KB)")

zip경로 = shutil.make_archive("/content/samples", "zip", OUT_DIR)
print(f"\n묶음 완성: {zip경로}  ({os.path.getsize(zip경로)/1024:.0f} KB)")

# --- 방법 B: 구글 드라이브로 복사 (가장 확실하다. 쓰려면 아래 세 줄의 # 을 지운다) ---
# from google.colab import drive
# drive.mount("/content/drive")
# shutil.copy(zip경로, "/content/drive/MyDrive/samples.zip"); print("드라이브에 복사 완료")

# --- 방법 C: 자동 다운로드 (브라우저 Colab 에서만 동작한다) ---
try:
    from google.colab import files
    files.download(zip경로)
except Exception as e:
    print(f"\n자동 다운로드 실패: {e}")
    print("→ VS Code 에서는 원래 안 됩니다. 왼쪽 Colab 파일 목록에서 samples.zip 을 받거나,")
    print("  위 '방법 B' 세 줄의 # 을 지우고 이 셀을 다시 실행하세요.")

## [셀 11] (선택) 버튼 달린 UI

여기까지로 도구는 완성이다. 이 셀은 덤으로, 매번 셀을 고쳐 실행하는 대신 **웹 화면에서 사진을 올리고 프롬프트를 쳐서** 바로 만들어 보는 UI 다.

실행하면 `gradio.live` 로 끝나는 임시 주소가 나온다. 이 셀이 돌아가는 동안만 유효하다.

> 이 주소는 아는 사람 누구나 들어올 수 있다. 수업이 끝나면 셀을 멈춰 두는 게 좋다.

In [ ]:
# [셀 11] (선택) 사진 올리고 프롬프트 쳐서 바로 만들어 보는 웹 UI.
import gradio as gr
import numpy as np

def ui_생성(참조사진, 프롬프트, 네거티브, 모드, 가로, 세로, 스텝, 시드, cfg):
    if 참조사진 is None:
        return None, None, "참조 사진을 올려주세요."
    try:
        태그 = str(int(time.time()))
        원본_ui = Image.fromarray(참조사진).convert("RGB")
        파일_원본 = f"ui_ref_{태그}.png"
        원본_ui.save(os.path.join(INPUT_DIR, 파일_원본))

        # 1) 관절 추출
        뼈대_ui = comfy_run({
            "1": {"class_type": "LoadImage", "inputs": {"image": 파일_원본}},
            "2": {"class_type": "OpenposePreprocessor",
                  "inputs": {"image": ["1", 0], "detect_body": "enable",
                             "detect_hand": "enable", "detect_face": "disable",
                             "resolution": 1024}},
            "3": {"class_type": "SaveImage",
                  "inputs": {"images": ["2", 0], "filename_prefix": "ui_skel"}},
        }, 결과노드="3")[0]

        파일_뼈대 = f"ui_skel_{태그}.png"
        뼈대_ui.save(os.path.join(INPUT_DIR, 파일_뼈대))
        참조 = 파일_뼈대 if 모드 == "뼈대" else 파일_원본

        # 2) 생성
        결과_ui = comfy_run({
            "10": {"class_type": "UnetLoaderGGUF", "inputs": {"unet_name": "flux-2-klein-4b-Q4_K_M.gguf"}},
            "8":  {"class_type": "CLIPLoader", "inputs": {"clip_name": "qwen_3_4b.safetensors", "type": "flux2"}},
            "4":  {"class_type": "VAELoader", "inputs": {"vae_name": "flux2-vae.safetensors"}},
            "12": {"class_type": "LoadImage", "inputs": {"image": 참조}},
            "14": {"class_type": "VAEEncode", "inputs": {"pixels": ["12", 0], "vae": ["4", 0]}},
            "6":  {"class_type": "CLIPTextEncode", "inputs": {"text": 프롬프트, "clip": ["8", 0]}},
            "15": {"class_type": "ReferenceLatent", "inputs": {"conditioning": ["6", 0], "latent": ["14", 0]}},
            "7":  {"class_type": "CLIPTextEncode", "inputs": {"text": 네거티브 or "", "clip": ["8", 0]}},
            "5":  {"class_type": "EmptyLatentImage",
                   "inputs": {"width": int(가로), "height": int(세로), "batch_size": 1}},
            "3":  {"class_type": "KSampler",
                   "inputs": {"seed": int(시드), "steps": int(스텝), "cfg": float(cfg),
                              "sampler_name": "euler", "scheduler": "simple", "denoise": 1.0,
                              "model": ["10", 0], "positive": ["15", 0],
                              "negative": ["7", 0], "latent_image": ["5", 0]}},
            "9":  {"class_type": "VAEDecode", "inputs": {"samples": ["3", 0], "vae": ["4", 0]}},
            "11": {"class_type": "SaveImage", "inputs": {"images": ["9", 0], "filename_prefix": "ui_out"}},
        }, 결과노드="11")[0]

        설명 = f"완료 — 시드 {int(시드)} / {int(스텝)}스텝 / cfg {cfg} / 참조 {모드}"
        return np.array(뼈대_ui), np.array(결과_ui), 설명
    except Exception as e:
        return None, None, f"실패: {e}"


with gr.Blocks() as demo:
    gr.Markdown("# 원하는 포즈로 이미지 만들기\n참조 사진을 올리고 프롬프트를 쓰면, 같은 자세의 다른 장면을 만듭니다.")
    with gr.Row():
        with gr.Column(scale=3):
            입력사진 = gr.Image(label="참조 사진 (자세를 가져올 사람)")
            입력프롬프트 = gr.Textbox(label="프롬프트", value=PROMPT, lines=3)
            입력네거티브 = gr.Textbox(label="네거티브 (선택)", value=NEGATIVE, lines=1)
            버튼 = gr.Button("만들기", variant="primary")
            상태 = gr.Markdown("")
        with gr.Column(scale=2):
            입력모드 = gr.Radio(["뼈대", "원본"], value="뼈대", label="무엇을 참조로 넣을지")
            입력가로 = gr.Slider(512, 1536, value=WIDTH, step=64, label="가로")
            입력세로 = gr.Slider(512, 1536, value=HEIGHT, step=64, label="세로")
            입력스텝 = gr.Slider(1, 8, value=STEPS, step=1, label="스텝 (4 권장)")
            입력cfg = gr.Slider(1.0, 6.0, value=CFG, step=0.5, label="CFG")
            입력시드 = gr.Number(value=SEED, label="시드 (고정하면 같은 결과)")
    with gr.Row():
        출력뼈대 = gr.Image(label="뽑아낸 뼈대")
        출력결과 = gr.Image(label="결과")

    버튼.click(ui_생성,
              [입력사진, 입력프롬프트, 입력네거티브, 입력모드,
               입력가로, 입력세로, 입력스텝, 입력시드, 입력cfg],
              [출력뼈대, 출력결과, 상태])

demo.launch(share=True, server_port=7860, quiet=True)

## 무엇을 바꿔 보았고, 어떻게 달라졌는가

> 실험을 마친 뒤 아래를 채운다. 루브릭 3번(스스로 평가)이 보는 곳이다.

### 1) 같은 포즈 + 프롬프트만 바꿈

| 프롬프트 | 시드 | 결과 | 포즈가 유지됐나 |
|---|---|---|---|
|  |  |  |  |
|  |  |  |  |

- 관찰: 

### 2) 같은 프롬프트 + 포즈 사진만 바꿈

| 참조 사진 | 시드 | 결과 | 포즈가 유지됐나 |
|---|---|---|---|
|  |  |  |  |
|  |  |  |  |

- 관찰: 

### 3) 참조모드 비교 (뼈대 vs 원본)

- 뼈대: 
- 원본: 

### 4) 재현 확인

- 같은 시드·같은 설정으로 다시 돌린 결과: 

### 5) 알게 된 것 / 한계

- 